# OKR paper (arXiv 2607.00185): Validations, Sensitivity Matrix & B7 Audit (v2)

**Run all cells top to bottom** in an environment created from `requirements.txt` (no auto `pip install`). Edit only the `CONFIG` cells or set `KHIPU_DB_PATH`.

Includes:
- **Baseline & Sensitivity Matrix** (`v1-exact` 27/619, `v1-corregida` 27/612, `robustez` 23/612, plus **`missing-aware`** comparison) reporting silhouette in **both** UMAP 2D space and original standardized feature space (`sil_FEAT`).
- **A1–A3** (clean `REGION` audit, `Late Horizon = Dallas` crosstab, SHAP naming, cord-level `TWIST` S/Z/U/blank table correcting v1's 85.3%).
- **B1–B6** (size control, null models, cataloger/documentation confound, grouped CV by **cataloger** and **institution (`MUSEUM_NAME`)**, imperial ablations with `<3`-positive guard, knot-type bigrams).
- **B7** (`CREATED_BY = jclindaniel` audit, `excluded_khipus_okr_2.1.0.csv`, `612 / 467 / 430` progression, threshold sweep `0.05/0.10/0.20/0.50`, depth-2 `DOC` tree, 18-config hyperparameter sweep, and `1,000 + 1,000` nulls in both UMAP & feature space).
- **Santa Valley replication** (`WHERE OKR_NUM BETWEEN 'KH0323' AND 'KH0328'`, no hardcoded `KHIPU_ID`s).


In [1]:
import os, sys
# Must be set BEFORE importing numpy/scipy/umap/numba to prevent thread oversubscription
for _v in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS', 'NUMBA_NUM_THREADS'):
    os.environ.setdefault(_v, '1')

import re, json, zipfile, sqlite3, warnings, traceback
from pathlib import Path
import numpy as np
import pandas as pd
import requests
import umap
import hdbscan
import xgboost as xgb
import shap
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import entropy as scipy_entropy, chi2_contingency
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import silhouette_score, adjusted_rand_score, f1_score
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold, cross_val_score, cross_val_predict
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.feature_extraction.text import TfidfVectorizer

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
from src.b7_nulls import run_null_suite, format_p, sil_feat

warnings.filterwarnings('ignore')
SEED = 42
np.random.seed(SEED)
print({k: getattr(__import__(k), '__version__', 'n/a') for k in ['numpy', 'pandas', 'scipy', 'umap', 'hdbscan', 'xgboost', 'shap', 'sklearn']})

STATUS = {}
class step:
    def __init__(self, name): self.name = name
    def __enter__(self): print(f'\n===== {self.name} ====='); return self
    def __exit__(self, et, ev, tb):
        STATUS[self.name] = 'OK' if et is None else f'FAILED: {et.__name__}: {str(ev)[:180]}'
        if et is not None:
            print('!! step failed ->', STATUS[self.name])
            traceback.print_exception(et, ev, tb, limit=3)
        return et is not None


C:\Users\maria\.gemini\antigravity\brain\fc73bd56-8b09-40d2-8033-004b16e8895b\scratch\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'numpy': '2.5.5', 'pandas': '3.1.1', 'scipy': '1.18.1', 'umap': '0.6.0', 'hdbscan': 'n/a', 'xgboost': '3.4.1', 'shap': '0.54.1', 'sklearn': '1.9.1'}


In [2]:
# ===================== CONFIG: edit only this cell =====================
RUN_MODE       = 'matrix'    # 'matrix' | 'v1-exact' | 'v1-corregida' | 'robustez'
PRIMARY_MODE   = 'v1-exact'  # detailed tables/figures/exports when RUN_MODE == 'matrix'

FEATURES_CSV   = None        # Optional CSV path (used only when RUN_MODE != 'matrix'); None = build from OKR SQLite
DB_PATH        = os.environ.get('KHIPU_DB_PATH', None)  # Single DB_PATH; falls back to data/ or Zenodo download
ZENODO_ID      = 18025748    # OKR record (DOI 10.5281/zenodo.18025748)
OKR_VERSION    = '2.1.0'     # Zenodo release tag

RESULTS_DIR    = str(REPO_ROOT / 'results')  # Versioned small CSV tables
OUTPUT_DIR     = str(REPO_ROOT / 'outputs')  # Gitignored figures & full JSON

ID_COL         = None
REGION_COL     = None
MUSEUM_COL     = None
NAME_COL       = None
IMPERIAL_LABEL = None
COLL_CLUSTER   = None

UMAP_KW        = dict(n_neighbors=15, min_dist=0.1, metric='euclidean')
HDB_KW         = dict(min_cluster_size=10, min_samples=5, metric='euclidean')
N_SIM          = 100
N_PERM         = 100
RUN_NULLS      = False       # Set True to recompute B2 in matrix; full N=1000+1000 runs in B7.5
RUN_PERMS      = False       # Set True to recompute B5 N_PERM=100 permutations (~35 min across 3 modes)
# ========================================================================
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)


## 1. Data Loading, Feature Builders, Expanded Missingness (`DOC`) & Role-Based Column Mapping


In [3]:
ROLE_SCHEMAS = {
    'v1': {
        'twist_ratio':   ['ratio_SZ'],
        'twist_and_dir': ['twist_S', 'twist_Z', 'ratio_SZ', 'knot_dir_S', 'knot_dir_Z', 'ratio_knot_SZ'],
        'geometry':      ['n_cords', 'mean_length', 'std_length', 'max_cord_level', 'ratio_subsidiary', 'n_knots', 'mean_turns'],
        'size':          ['n_cords', 'n_knots'],
    },
    'v2': {
        'twist_ratio':   ['twist_ratio'],
        'twist_and_dir': ['n_twist_S', 'n_twist_Z', 'twist_ratio', 'knot_dir_S', 'knot_dir_Z'],
        'geometry':      ['n_cords', 'cord_len_mean', 'cord_len_std', 'max_level', 'subsidiary_ratio', 'n_knots', 'knot_turns_mean'],
        'size':          ['n_cords', 'n_knots'],
    }
}
EXPECTED_COUNTS = {
    'v1': {'twist_ratio': 1, 'twist_and_dir': 6, 'geometry': 7, 'size': 2},
    'v2': {'twist_ratio': 1, 'twist_and_dir': 5, 'geometry': 7, 'size': 2},
}

def resolve_roles(columns, verbose=True):
    cols = set(columns)
    scores = {sname: len(cols & set().union(*rdict.values())) for sname, rdict in ROLE_SCHEMAS.items()}
    schema = max(scores, key=scores.get)
    rdict = ROLE_SCHEMAS[schema]
    assigned = {}
    for role, expected in rdict.items():
        present = [c for c in expected if c in cols]
        missing = [c for c in expected if c not in cols]
        exp_n = EXPECTED_COUNTS[schema][role]
        assert len(present) == exp_n and not missing, (
            f"Incomplete role '{role}' under schema '{schema}': "
            f"expected {exp_n} cols {expected}, found {len(present)} ({present}), missing {missing}"
        )
        assigned[role] = present
    if verbose:
        print(f"[resolve_roles] Detected schema: {schema}")
        for role, clist in assigned.items():
            print(f"  {role:15s} ({len(clist)}): {clist}")
    return schema, assigned

def pick(cols, cands):
    low = {c.lower(): c for c in cols}
    for c in cands:
        if c.lower() in low:
            return low[c.lower()]
    return None

def open_db(db_path=None):
    path = db_path or DB_PATH
    if not path:
        for root in [str(REPO_ROOT / 'data'), '.', '..', 'data', 'okr_data']:
            if os.path.isdir(root):
                for d, _, fs in os.walk(root):
                    for f in fs:
                        if f.lower().endswith(('.sqlite', '.sqlite3', '.db')) and 'khipu' in f.lower():
                            path = os.path.join(d, f); break
                    if path: break
            if path: break
    if not path:
        rec = requests.get(f'https://zenodo.org/api/records/{ZENODO_ID}', timeout=60).json()
        cand = [(f.get('key') or f.get('filename') or '', (f.get('links') or {}).get('self') or (f.get('links') or {}).get('download'))
                for f in rec.get('files', []) if (f.get('key') or f.get('filename') or '').lower().endswith(('.sqlite', '.sqlite3', '.db', '.zip'))]
        assert cand, 'No sqlite/zip in the Zenodo record: download manually and set KHIPU_DB_PATH'
        cand.sort(key=lambda t: not t[0].lower().endswith(('.sqlite', '.sqlite3', '.db')))
        name, url = cand[0]
        os.makedirs('okr_data', exist_ok=True)
        dst = os.path.join('okr_data', name)
        if not os.path.exists(dst):
            r = requests.get(url, timeout=600); r.raise_for_status(); open(dst, 'wb').write(r.content)
        if dst.lower().endswith('.zip'):
            zipfile.ZipFile(dst).extractall('okr_data')
            dbs = [os.path.join(d, f) for d, _, fs in os.walk('okr_data') for f in fs if f.lower().endswith(('.sqlite', '.sqlite3', '.db'))]
            assert dbs, 'zip has no sqlite'; dst = dbs[0]
        path = dst
    print('DB:', path)
    return sqlite3.connect(path)

def q(con, table):
    names = {n.lower(): n for (n,) in con.execute("select name from sqlite_master where type in ('table','view')")}
    t = names.get(table.lower())
    assert t, f'table {table} not found; have {list(names.values())}'
    df = pd.read_sql(f'select * from "{t}"', con)
    df.columns = [c.lower() for c in df.columns]
    return df

def _doc_features_from_db(con):
    """Expanded documentation/missingness features (P1.4 & P2.1: separate U-code vs blank/null)."""
    cord = q(con, 'cord')
    knot = q(con, 'knot')
    colors = q(con, 'ascher_cord_color')
    knot_m = knot.merge(cord[['cord_id', 'khipu_id']], on='cord_id', how='inner')

    tw = cord['twist'].fillna('').astype(str).str.upper().str.strip()
    kd = knot_m['direction'].fillna('').astype(str).str.upper().str.strip()
    att = cord['attachment_type'].fillna('').astype(str).str.upper().str.strip()
    cord_cnt = cord.groupby('khipu_id').size()

    D = pd.DataFrame(index=cord_cnt.index)
    D['pct_twist_Ucode']     = (tw == 'U').groupby(cord['khipu_id']).mean()
    D['pct_twist_blank']     = (tw == '').groupby(cord['khipu_id']).mean()
    D['pct_twist_U']         = (~tw.isin(['S', 'Z'])).groupby(cord['khipu_id']).mean()
    D['pct_no_length']       = cord['cord_length'].isna().groupby(cord['khipu_id']).mean()
    D['color_rows_per_cord'] = (colors.groupby('khipu_id').size() / cord_cnt).fillna(0.0)
    D['knot_rows_per_cord']  = (knot_m.groupby('khipu_id').size() / cord_cnt).fillna(0.0)
    D['pct_knot_dir_U']      = (~kd.isin(['S', 'Z'])).groupby(knot_m['khipu_id']).mean()
    D['pct_side_U']          = (~att.isin(['R', 'V'])).groupby(cord['khipu_id']).mean()
    D.index = D.index.astype(str)
    D.index.name = 'khipu_id'
    return D

def _finalize_doc(D_raw, index):
    DOC = D_raw.reindex(index).copy()
    for c in ['pct_twist_Ucode', 'pct_twist_blank', 'pct_twist_U', 'pct_no_length', 'pct_knot_dir_U', 'pct_side_U']:
        if c in DOC.columns:
            DOC[c] = DOC[c].fillna(1.0)
    for c in ['color_rows_per_cord', 'knot_rows_per_cord']:
        if c in DOC.columns:
            DOC[c] = DOC[c].fillna(0.0)
    return DOC

def build_v1_from_db(con, drop_empty=False, fillna_zero=True):
    """Exact replication of 01_pipeline_khipu.ipynb (27 features); fillna_zero=False preserves NaN for missing-aware mode."""
    kh = q(con, 'khipu_main'); cord = q(con, 'cord'); knot = q(con, 'knot'); colors = q(con, 'ascher_cord_color')
    cord_feats = cord.groupby('khipu_id').agg(
        n_cords          = ('cord_id', 'count'),
        mean_length      = ('cord_length', 'mean'),
        std_length       = ('cord_length', 'std'),
        max_cord_level   = ('cord_level', 'max'),
        n_subsidiary     = ('cord_level', lambda x: (x > 1).sum()),
        ratio_subsidiary = ('cord_level', lambda x: (x > 1).mean()),
        twist_S          = ('twist', lambda x: (x == 'S').sum()),
        twist_Z          = ('twist', lambda x: (x == 'Z').sum()),
    ).reset_index()
    denom_tw = cord_feats['twist_S'] + cord_feats['twist_Z']
    cord_feats['ratio_SZ'] = (cord_feats['twist_S'] / (denom_tw + 1e-9)) if fillna_zero else (cord_feats['twist_S'] / denom_tw.replace(0, np.nan))
    if fillna_zero:
        cord_feats['std_length'] = cord_feats['std_length'].fillna(0)

    merged = knot.merge(cord[['cord_id', 'khipu_id']], on='cord_id', how='left')
    type_dummies = pd.get_dummies(merged['type_code'], prefix='knot')
    knot_type_feats = pd.concat([merged[['khipu_id']], type_dummies], axis=1).groupby('khipu_id').sum().reset_index()
    knot_general = merged.groupby('khipu_id').agg(
        n_knots    = ('type_code', 'count'),
        mean_turns = ('num_turns', 'mean'),
        knot_dir_S = ('direction', lambda x: (x == 'S').sum()),
        knot_dir_Z = ('direction', lambda x: (x == 'Z').sum()),
    ).reset_index()
    denom_kn = knot_general['knot_dir_S'] + knot_general['knot_dir_Z']
    knot_general['ratio_knot_SZ'] = (knot_general['knot_dir_S'] / (denom_kn + 1e-9)) if fillna_zero else (knot_general['knot_dir_S'] / denom_kn.replace(0, np.nan))
    knot_feats = knot_general.merge(knot_type_feats, on='khipu_id', how='left')

    def _color_ent(s):
        c = s.dropna().value_counts(normalize=True)
        return float(scipy_entropy(c)) if len(c) > 1 else 0.0
    color_feats = colors.groupby('khipu_id').agg(
        n_unique_colors = ('color_cd_1', 'nunique'),
        color_entropy   = ('color_cd_1', _color_ent),
        has_multicolor  = ('color_cd_2', lambda x: int(x.notna().any())),
    ).reset_index()

    fm = kh[['khipu_id']].merge(cord_feats, on='khipu_id', how='left').merge(knot_feats, on='khipu_id', how='left').merge(color_feats, on='khipu_id', how='left')
    fm = fm.set_index('khipu_id'); fm.index = fm.index.astype(str); fm.index.name = 'khipu_id'
    if fillna_zero:
        fm = fm.fillna(0)
    if drop_empty:
        fm = fm[fm['n_cords'].fillna(0) > 0].copy()
    D = _doc_features_from_db(con)
    return fm, D, kh

def build_v2_from_db(con):
    """Reconstructed 23-feature matrix (starting from cord, n=612)."""
    kh, cord, knot = q(con, 'khipu_main'), q(con, 'cord'), q(con, 'knot')
    knot = knot.merge(cord[['cord_id', 'khipu_id']], on='cord_id', how='inner')
    g = cord.groupby('khipu_id')
    F = pd.DataFrame(index=g.size().index)
    F['n_cords'] = g.size()
    F['cord_len_mean'] = g['cord_length'].mean(); F['cord_len_std'] = g['cord_length'].std()
    lv = pd.to_numeric(cord['cord_level'], errors='coerce')
    F['max_level'] = lv.groupby(cord['khipu_id']).max()
    F['subsidiary_ratio'] = (lv > 1).groupby(cord['khipu_id']).mean()
    tw = cord['twist'].astype(str).str.upper().str.strip()
    F['n_twist_S'] = (tw == 'S').groupby(cord['khipu_id']).sum(); F['n_twist_Z'] = (tw == 'Z').groupby(cord['khipu_id']).sum()
    F['twist_ratio'] = F['n_twist_S'] / (F['n_twist_S'] + F['n_twist_Z']).replace(0, np.nan)
    gk = knot.groupby('khipu_id')
    F['n_knots'] = gk.size()
    F['knot_turns_mean'] = gk['num_turns'].apply(lambda s: pd.to_numeric(s, errors='coerce').mean())
    kd = knot['direction'].astype(str).str.upper().str.strip()
    F['knot_dir_S'] = (kd == 'S').groupby(knot['khipu_id']).sum(); F['knot_dir_Z'] = (kd == 'Z').groupby(knot['khipu_id']).sum()
    ct = pd.crosstab(knot['khipu_id'], knot['type_code'].astype(str))
    ct = ct[ct.sum().sort_values(ascending=False).index[:8]]
    ct.columns = [f'knot_type_{c}' for c in ct.columns]
    F = F.join(ct)
    col = q(con, 'ascher_cord_color').merge(cord[['cord_id', 'khipu_id']], on='cord_id', how='inner', suffixes=('', '_c'))
    kk = 'khipu_id' if 'khipu_id' in col.columns else 'khipu_id_c'
    def _ent(s):
        p = s.value_counts(normalize=True).values
        return float(-(p * np.log(p)).sum())
    F['n_colors'] = col.groupby(kk)['color_cd_1'].nunique()
    F['color_entropy'] = col.groupby(kk)['color_cd_1'].apply(_ent)
    F['multicolor'] = (F['n_colors'] > 1).astype(int)
    F.index = F.index.astype(str); F.index.name = 'khipu_id'
    D = _doc_features_from_db(con)
    return F.fillna(0), D, kh

def load_dataset(mode, con, features_csv=None, verbose=True):
    if features_csv is not None:
        raw = pd.read_csv(features_csv)
        id_c = ID_COL or pick(raw.columns, ['khipu_id', 'id', 'okr_id', 'khipu_name', 'name']) or raw.columns[0]
        raw = raw.set_index(id_c); raw.index = raw.index.astype(str); raw.index.name = 'khipu_id'
        raw = raw[[c for c in raw.columns if not re.match(r'^unnamed', c, re.I)]]
        csv_cl = [c for c in raw.columns if re.search(r'cluster|hdbscan|umap|^label', c, re.I)]
        D_csv = raw[[c for c in raw.columns if re.search(r'(^pct_.*_u$|unrecorded|pct_.*unknown)', c, re.I)]].copy()
        META = raw.select_dtypes(exclude=[np.number]).copy()
        F = raw.select_dtypes(include=[np.number]).drop(columns=list(D_csv.columns) + csv_cl, errors='ignore')
        kh = q(con, 'khipu_main')
        D = D_csv if D_csv.shape[1] > 0 else _doc_features_from_db(con)
    elif mode == 'v1-exact':
        F, D, kh = build_v1_from_db(con, drop_empty=False, fillna_zero=True); META = pd.DataFrame(index=F.index)
    elif mode == 'v1-corregida':
        F, D, kh = build_v1_from_db(con, drop_empty=True, fillna_zero=True); META = pd.DataFrame(index=F.index)
    elif mode == 'robustez':
        F, D, kh = build_v2_from_db(con); META = pd.DataFrame(index=F.index)
    else:
        raise ValueError(f'Unknown mode: {mode}')

    idx_s = set(map(str, F.index))
    best_key = max(kh.columns, key=lambda c: len(idx_s & set(kh[c].astype(str))))
    kh_idx = kh.drop_duplicates(best_key).set_index(best_key); kh_idx.index = kh_idx.index.astype(str)
    meta_overlap = len(idx_s & set(kh_idx.index)) / len(F)
    assert meta_overlap >= 0.95, f"Metadata join coverage too low: {meta_overlap:.1%} (<95%)"

    META = META.join(kh_idx[[c for c in kh_idx.columns if c not in META.columns]], how='left')
    DOC_raw = D.reindex(F.index)
    doc_coverage = float(DOC_raw.notna().any(axis=1).mean())
    assert doc_coverage >= 0.95, f"Documentation join coverage too low: {doc_coverage:.1%} (<95%)"

    DOC = _finalize_doc(DOC_raw, F.index)
    schema, roles = resolve_roles(F.columns, verbose=verbose)
    if verbose:
        print(f"[{mode}] n={len(F)}, p={F.shape[1]}, meta_cov={meta_overlap:.1%}, doc_cov={doc_coverage:.1%}")
    return F, DOC, META, schema, roles


## 2. Full Validation Engine (Baseline, A1–A3, B1–B6)


In [4]:
def run_cluster(Z, seed=SEED):
    emb = umap.UMAP(n_components=2, random_state=seed, **UMAP_KW).fit_transform(Z)
    lab = hdbscan.HDBSCAN(**HDB_KW).fit_predict(emb)
    ok = lab >= 0
    s_umap = float(silhouette_score(emb[ok], lab[ok])) if ok.sum() > 1 and len(set(lab[ok])) > 1 else np.nan
    s_feat = sil_feat(Z, lab)
    return emb, lab, s_umap, s_feat

def cramers_v(ct):
    chi2 = chi2_contingency(ct)[0]; n_tot = ct.values.sum(); k_min = min(ct.shape) - 1
    return float(np.sqrt(chi2 / (n_tot * k_min))) if k_min > 0 else np.nan

def clean_region_series(series):
    """P2.5: strip quotes and whitespace; filter blank/unknown/NA consistently across pandas 2.x/3.x."""
    reg = series.fillna('').astype(str).str.strip().str.strip('"').str.strip()
    bad = {'', ' ', 'nan', 'none', 'unknown', 'na', 'n/a', 'null', '<na>'}
    assert ' ' in bad and '' in bad
    return reg[~reg.str.lower().isin(bad)]

def build_bigram_df(con):
    knot_seq = pd.read_sql('''
        SELECT k.CORD_ID, c.KHIPU_ID, k.TYPE_CODE, k.KNOT_ORDINAL
        FROM knot k JOIN cord c ON k.CORD_ID = c.CORD_ID
        WHERE k.TYPE_CODE IS NOT NULL AND k.KNOT_ORDINAL IS NOT NULL
        ORDER BY c.KHIPU_ID, k.CORD_ID, k.KNOT_ORDINAL
    ''', con)
    seqs = knot_seq.groupby('KHIPU_ID')['TYPE_CODE'].apply(list).reset_index()
    seqs['bigram_text'] = seqs['TYPE_CODE'].apply(lambda s: ' '.join(f'{s[i]}-{s[i+1]}' for i in range(len(s) - 1)))
    tfidf = TfidfVectorizer(max_features=50, min_df=5)
    X_ng = tfidf.fit_transform(seqs['bigram_text']).toarray()
    ng_df = pd.DataFrame(X_ng, columns=[f'ngram_{f}' for f in tfidf.get_feature_names_out()], index=seqs['KHIPU_ID'].astype(str))
    ng_df.index.name = 'khipu_id'
    return ng_df

def run_full_analysis(mode, con, features_csv=None, run_nulls=RUN_NULLS, n_sim=N_SIM, n_perm=N_PERM, verbose=True):
    R = {'mode': mode}
    F, DOC, META, schema, roles = load_dataset(mode, con, features_csv=features_csv, verbose=verbose)
    R['n'] = int(len(F)); R['n_features'] = int(F.shape[1]); R['schema'] = schema

    REGION = REGION_COL or pick(META.columns, ['region', 'geographic_region', 'provenance_region', 'provenance'])
    MUSEUM = MUSEUM_COL or pick(META.columns, ['museum_name', 'museum', 'museum_collection'])
    NAME   = NAME_COL   or pick(META.columns, ['investigator_num', 'khipu_name', 'orig_khipu_name', 'kfg_name', 'name', 'nickname'])
    CATALOGER = META[NAME].astype(str).str.extract(r'^([A-Za-z]{2})')[0].str.upper() if NAME else pd.Series(np.nan, index=F.index)

    # ---- Baseline clustering (UMAP + Feature space silhouette) ----
    Z0 = StandardScaler().fit_transform(F.values)
    emb0, lab0, sil0_u, sil0_f = run_cluster(Z0)
    sizes = pd.Series(lab0).value_counts().sort_index()
    non_noise_sizes = sorted([int(v) for k, v in sizes.items() if k >= 0])
    R['baseline'] = dict(
        n_clusters=int((sizes.index >= 0).sum()),
        sil_umap=float(sil0_u), sil_feat=float(sil0_f),
        sizes={int(k): int(v) for k, v in sizes.items()},
        sorted_sizes=non_noise_sizes,
    )
    if verbose:
        print(f"[{mode}] Baseline clusters: {sizes.to_dict()} | sil_UMAP={sil0_u:.3f} | sil_FEAT={sil0_f:.3f}")
    if mode == 'v1-exact':
        ref_sizes = [17, 160, 442]
        sizes_ok = (len(non_noise_sizes) == 3 and all(abs(a - b) <= 5 for a, b in zip(non_noise_sizes, ref_sizes)))
        if sizes_ok and abs(sil0_u - 0.769) <= 0.08:
            print(f"[v1-exact] Reproduces v1 within UMAP tolerance (got {non_noise_sizes}, sil_UMAP={sil0_u:.3f}, sil_FEAT={sil0_f:.3f})")

    CL = pd.Series(lab0, index=F.index, name='cluster')
    big = [c for c in sizes.index if c >= 0]
    order = sizes.loc[big].sort_values(ascending=False).index.tolist()
    MAIN_CLUSTER = order[0]
    COLL_CL = COLL_CLUSTER if COLL_CLUSTER is not None else (order[1] if len(order) > 1 else order[0])

    # ---- A1: Label audit (clean quotes/spaces) + P1.2 Late Horizon = Dallas ----
    nonnull = clean_region_series(META[REGION]) if REGION else pd.Series([], dtype=str)
    vc = nonnull.value_counts()
    vc10 = nonnull.where(nonnull.map(vc) >= 10, 'Other')
    imp_label = IMPERIAL_LABEL or ([l for l in vc.index if 'inka' in l.lower() and 'late' in l.lower()][0])
    idx = nonnull.index.intersection(F.index)
    R['A1'] = dict(
        total=int(len(F)), has_value=int(META[REGION].notna().sum()) if REGION else 0,
        labeled=int(len(nonnull)), n_used=int(len(idx)),
        classes_ge10=int((vc >= 10).sum()), excl_other=int((vc10 != 'Other').sum()),
        imperial_label=imp_label,
    )
    if verbose and mode == PRIMARY_MODE:
        print('\n[P1.2] Crosstab (Inka, Late Horizon vs MUSEUM_NAME):')
        print(pd.crosstab(nonnull.loc[idx] == imp_label, META.loc[idx, MUSEUM].fillna('NA').str.strip().str.strip('"')).T.sort_values(True, ascending=False).head(6))

    # ---- A2: SHAP naming ----
    y_imp = (nonnull.loc[idx] == imp_label).astype(int).values
    Xl = F.loc[idx]
    m_a2 = xgb.XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05, subsample=0.8,
                             colsample_bytree=0.8, random_state=SEED, eval_metric='logloss', n_jobs=1).fit(Xl, y_imp)
    sv = shap.TreeExplainer(m_a2).shap_values(Xl)
    sv = sv[1] if isinstance(sv, list) else sv
    imp_s = pd.Series(np.abs(sv).mean(0), index=Xl.columns).sort_values(ascending=False)
    top10 = imp_s.head(10)
    def _kind(n):
        if 'twist' in n or n == 'ratio_SZ': return 'CORD twist'
        if 'knot_dir' in n or n == 'ratio_knot_SZ': return 'KNOT direction'
        return 'other'
    R['A2'] = dict(top_features=list(top10.index), top_values=[float(x) for x in top10.values], kinds=[_kind(n) for n in top10.index])

    # ---- A3: Unrecorded twist test ----
    tr_col = roles['twist_ratio'][0]
    def rank_biserial(a, b):
        u = stats.mannwhitneyu(a, b, alternative='two-sided')
        return float(u.pvalue), float(1 - 2 * u.statistic / (len(a) * len(b)))
    A = pd.DataFrame({'cl': CL, 'twist_ratio': F[tr_col], 'pct_twist_U': DOC['pct_twist_U']})
    a = A[A.cl == COLL_CL]; b = A[A.cl == MAIN_CLUSTER]
    p_all, rb_all = rank_biserial(a.twist_ratio.dropna(), b.twist_ratio.dropna())
    ra = a[a.pct_twist_U < 0.2].twist_ratio.dropna(); rbm = b[b.pct_twist_U < 0.2].twist_ratio.dropna()
    p_rec, rb_rec = rank_biserial(ra, rbm) if len(ra) > 5 and len(rbm) > 5 else (np.nan, np.nan)
    R['A3'] = dict(twist_col=tr_col, p_all=p_all, rb_all=rb_all, p_recorded=p_rec, rb_recorded=rb_rec,
                   n_rec=(int(len(ra)), int(len(rbm))), unrec_by_cluster=A.groupby('cl')['pct_twist_U'].mean().round(3).to_dict())

    # ---- B1: Size control ----
    size_cols = roles['size']
    S = np.log1p(F[size_cols].values)
    Xr = F.drop(columns=size_cols).copy()
    for c in Xr.columns:
        lr = LinearRegression().fit(S, F[c].values)
        Xr[c] = F[c].values - lr.predict(S)
    Z1 = StandardScaler().fit_transform(Xr.values)
    _, lab1, sil1_u, sil1_f = run_cluster(Z1)
    ok_b1 = (lab0 >= 0) & (lab1 >= 0)
    ari = float(adjusted_rand_score(lab0[ok_b1], lab1[ok_b1])) if ok_b1.sum() > 1 else np.nan
    R['B1'] = dict(n_clusters=int(len(set(lab1)) - (1 if -1 in lab1 else 0)), sil_umap=float(sil1_u), sil_feat=float(sil1_f), ari_vs_baseline=ari)

    # ---- B3: Cataloger + documentation confound ----
    ok = CL >= 0
    ct_cat = pd.crosstab(CL[ok], CATALOGER[ok].fillna('NA'))
    R['B3_cramers_v_cataloger'] = cramers_v(ct_cat)
    top_m = META[MUSEUM].fillna('NA').value_counts().head(8).index
    ct_m = pd.crosstab(CL[ok], META[MUSEUM].fillna('NA').where(META[MUSEUM].fillna('NA').isin(top_m), 'other')[ok])
    R['B3_cramers_v_museum'] = cramers_v(ct_m)
    Xd = np.column_stack([DOC[ok].fillna(1.0).values, pd.get_dummies(CATALOGER[ok].fillna('NA')).values])
    yd = CL[ok].values
    base_maj = float(pd.Series(yd).value_counts(normalize=True).max())
    acc_doc = float(cross_val_score(LogisticRegression(max_iter=2000), StandardScaler().fit_transform(Xd), yd, cv=5).mean())
    R['B3_doc_only_acc'] = dict(acc=acc_doc, majority=base_maj)

    # ---- B4: Grouped CV by cataloger AND by institution (P1.3 MUSEUM_NAME) ----
    def cv_f1(Xm, ym, groups=None, n=5):
        classes = np.unique(ym); enc = {c: i for i, c in enumerate(classes)}; yi = np.array([enc[c] for c in ym])
        if groups is not None:
            n_g = len(set(groups))
            n = min(n, n_g)
            if n < 3: return np.nan, np.nan
            splits = StratifiedGroupKFold(n_splits=n, shuffle=True, random_state=SEED).split(Xm, yi, groups)
        else:
            splits = StratifiedKFold(n_splits=n, shuffle=True, random_state=SEED).split(Xm, yi)
        pred = np.zeros(len(yi), int)
        for tr, te in splits:
            le = LabelEncoder().fit(yi[tr])
            if len(le.classes_) < 2:
                pred[te] = le.classes_[0]; continue
            mdl = xgb.XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05,
                                    subsample=0.8, colsample_bytree=0.8, random_state=SEED, n_jobs=1)
            mdl.fit(Xm.iloc[tr], le.transform(yi[tr]))
            pred[te] = le.inverse_transform(mdl.predict(Xm.iloc[te]))
        f1w = float(f1_score(yi, pred, average='weighted'))
        # P1.3: only evaluate imperial F1 under grouped CV if >=3 distinct groups have imperial positives
        if imp_label in enc:
            if groups is not None and len(set(groups[yi == enc[imp_label]])) < 3:
                f1i = float('nan')
            else:
                f1i = float(f1_score(yi == enc[imp_label], pred == enc[imp_label]))
        else:
            f1i = float('nan')
        return f1w, f1i

    lab_y = vc10.loc[idx].values
    grp_cat = CATALOGER.loc[idx].fillna('NA').values
    grp_mus = META.loc[idx, MUSEUM].fillna('NA').str.strip().str.strip('"').values
    b4_rows = []
    for fname, Xm in [('structural only', F.loc[idx]), ('structural + documentation', F.loc[idx].join(DOC.loc[idx]))]:
        for cvname, g in [('stratified (paper)', None), ('grouped by cataloger', grp_cat), ('grouped by museum (LOIO)', grp_mus)]:
            w, i = cv_f1(Xm.fillna(0), lab_y, g)
            b4_rows.append(dict(features=fname, cv=cvname, F1_weighted=round(w, 3) if not np.isnan(w) else np.nan,
                                F1_imperial=round(i, 3) if not np.isnan(i) else np.nan))
    R['B4'] = b4_rows

    # ---- B5: Imperial ablations with P2.6 (<3 positives -> NaN) guard ----
    lab_s = pd.Series(lab_y, index=idx); cat_s = CATALOGER.loc[idx].fillna('NA')
    y_all = (lab_s == imp_label).astype(int).values
    ct5 = pd.crosstab(lab_s, cat_s)
    R['B5'] = dict(
        n_imperial=int(y_all.sum()),
        imperial_by_cataloger={str(k): int(v) for k, v in ct5.loc[imp_label].items() if v > 0},
        cataloger_only_f1w=float(cv_f1(pd.get_dummies(cat_s).astype(float), lab_y)[0]),
    )
    def f1_bin(X, y):
        if int(y.sum()) < 3:
            return float('nan')
        k = int(min(5, y.sum()))
        m = xgb.XGBClassifier(n_estimators=200, max_depth=3, learning_rate=0.05,
                              subsample=0.8, colsample_bytree=0.8, random_state=SEED, n_jobs=1)
        return float(f1_score(y, cross_val_predict(m, X, y, cv=StratifiedKFold(k, shuffle=True, random_state=SEED))))
    def perm_test(X, y, n=n_perm):
        if int(y.sum()) < 3:
            return float('nan'), float('nan')
        obs = f1_bin(X, y)
        if not RUN_PERMS:
            return obs, float('nan')
        rng = np.random.default_rng(SEED)
        cnt = sum(f1_bin(X, rng.permutation(y)) >= obs for _ in range(n))
        return obs, float((cnt + 1) / (n + 1))

    sets = {
        'structure (all features)':        F.loc[idx].fillna(0),
        'structure w/o twist & direction': F.loc[idx].drop(columns=roles['twist_and_dir']).fillna(0),
        'geometry only':                   F.loc[idx][roles['geometry']].fillna(0),
        'documentation only':              DOC.loc[idx].fillna(1.0),
    }
    R['B5']['imperial_tests'] = {sname: dict(zip(['f1', 'p'], [round(x, 3) if not np.isnan(x) else np.nan for x in perm_test(Xs, y_all)])) for sname, Xs in sets.items()}

    # ---- B6: Knot-type bigrams ----
    ng_df = build_bigram_df(con)
    X_base = F.loc[idx].fillna(0)
    X_plus_ng = pd.concat([X_base, ng_df.reindex(idx).fillna(0)], axis=1)
    w_bs, _ = cv_f1(X_base, lab_y, None); w_ns, _ = cv_f1(X_plus_ng, lab_y, None)
    w_bg, _ = cv_f1(X_base, lab_y, grp_cat); w_ng, _ = cv_f1(X_plus_ng, lab_y, grp_cat)
    R['B6_bigrams'] = dict(
        n_rows=int(len(idx)), n_with_seq=int(idx.isin(ng_df.index).sum()),
        stratified=dict(base_f1w=round(w_bs, 3), plus_ng_f1w=round(w_ns, 3), delta_f1w=round(w_ns - w_bs, 3)),
        grouped=dict(base_f1w=round(w_bg, 3), plus_ng_f1w=round(w_ng, 3), delta_f1w=round(w_ng - w_bg, 3)),
    )
    return R, dict(F=F, DOC=DOC, META=META, CL=CL, CATALOGER=CATALOGER, emb0=emb0, lab0=lab0, ct_cat=ct_cat, top10=top10, nonnull=nonnull, imp_label=imp_label)


## 3. Sensitivity Matrix, Cord-Level `TWIST` Audit (P2.2) & `missing-aware` Comparison (P2.4)


In [5]:
con = open_db()
ALL_RESULTS = {}
PRIMARY_ART = None

modes_to_run = ['v1-exact', 'v1-corregida', 'robustez'] if RUN_MODE == 'matrix' else [RUN_MODE]
for m in modes_to_run:
    with step(f'Run {m}'):
        res, art = run_full_analysis(m, con, features_csv=(FEATURES_CSV if RUN_MODE != 'matrix' else None),
                                     run_nulls=RUN_NULLS, n_sim=N_SIM, n_perm=N_PERM, verbose=True)
        ALL_RESULTS[m] = res
        if m == (PRIMARY_MODE if RUN_MODE == 'matrix' else RUN_MODE):
            PRIMARY_ART = art

def _b4_val(R, feat, cv_prefix, col):
    for row in R.get('B4', []):
        if row['features'] == feat and row['cv'].startswith(cv_prefix):
            return row[col]
    return np.nan

sens_rows = []
for m, R in ALL_RESULTS.items():
    imp_t = R['B5']['imperial_tests']
    sens_rows.append({
        'Corrida': m, 'Features': R['n_features'], 'n': R['n'],
        'Clusters': str(R['baseline']['sorted_sizes']),
        'Sil_UMAP': round(R['baseline']['sil_umap'], 3),
        'Sil_FEAT': round(R['baseline']['sil_feat'], 3),
        'B1_ARI': round(R['B1']['ari_vs_baseline'], 2),
        'B3_V_cat': round(R['B3_cramers_v_cataloger'], 2),
        'B3_V_mus': round(R['B3_cramers_v_museum'], 2),
        'B3_doc_acc': round(R['B3_doc_only_acc']['acc'], 2),
        'A1_lbl': R['A1']['labeled'],
        'B4_F1w_strat': _b4_val(R, 'structural only', 'stratified', 'F1_weighted'),
        'B4_F1i_strat': _b4_val(R, 'structural only', 'stratified', 'F1_imperial'),
        'B4_F1w_cat': _b4_val(R, 'structural only', 'grouped by cataloger', 'F1_weighted'),
        'B4_F1w_mus': _b4_val(R, 'structural only', 'grouped by museum', 'F1_weighted'),
        'B5_n_imp': R['B5']['n_imperial'],
        'B5_all_F1': imp_t['structure (all features)']['f1'],
        'B5_geo_F1': imp_t['geometry only']['f1'],
        'B5_doc_F1': imp_t['documentation only']['f1'],
        'B6_dF1_strat': R['B6_bigrams']['stratified']['delta_f1w'],
    })
SENS_DF = pd.DataFrame(sens_rows)
SENS_DF.to_csv(os.path.join(RESULTS_DIR, 'table_sensitivity_matrix.csv'), index=False)
print('\n================ SENSITIVITY MATRIX ================')
print(SENS_DF.to_string(index=False))

with step('P2.2 Cord-level TWIST by cluster (correcting v1 85.3%)'):
    cord_raw = q(con, 'cord')[['khipu_id', 'twist']].copy()
    cord_raw['cl'] = cord_raw['khipu_id'].astype(str).map(PRIMARY_ART['CL'])
    tw_s = cord_raw['twist'].fillna('').astype(str).str.upper().str.strip()
    cord_raw['tw_cat'] = np.where(tw_s == 'S', 'S', np.where(tw_s == 'Z', 'Z', np.where(tw_s == 'U', 'U', 'blank_or_null')))
    ct_cnt = pd.crosstab(cord_raw['cl'], cord_raw['tw_cat'])
    ct_pct = (ct_cnt.div(ct_cnt.sum(axis=1), axis=0) * 100).round(1)
    ct_out = ct_cnt.join(ct_pct, lsuffix='_n', rsuffix='_pct')
    ct_out.to_csv(os.path.join(RESULTS_DIR, 'table_twist_by_cluster.csv'))
    print(ct_pct.to_string())

with step('P2.4 Missing-aware mode (NaN + completeness indicators) vs fillna(0)'):
    F_na, D_na_raw, _ = build_v1_from_db(con, drop_empty=True, fillna_zero=False)
    D_na = _finalize_doc(D_na_raw, F_na.index)
    # Clustering with median imputation (only for UMAP distance) + completeness indicators
    X_ma_imp = SimpleImputer(strategy='median').fit_transform(pd.concat([F_na, D_na], axis=1).values)
    Z_ma = StandardScaler().fit_transform(X_ma_imp)
    _, lab_ma, su_ma, sf_ma = run_cluster(Z_ma, seed=SEED)
    # Classification with native NaN in XGBoost (+ completeness indicators)
    nonnull_ma = clean_region_series(PRIMARY_ART['META'].loc[F_na.index, 'region'])
    vc_ma = nonnull_ma.value_counts(); vc10_ma = nonnull_ma.where(nonnull_ma.map(vc_ma) >= 10, 'Other')
    idx_ma = nonnull_ma.index
    X_xgb_ma = pd.concat([F_na.loc[idx_ma], D_na.loc[idx_ma]], axis=1)
    le_ma = LabelEncoder(); y_ma = le_ma.fit_transform(vc10_ma.loc[idx_ma])
    pred_ma = cross_val_predict(xgb.XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, random_state=SEED, n_jobs=1),
                                X_xgb_ma, y_ma, cv=StratifiedKFold(5, shuffle=True, random_state=SEED))
    f1w_ma = float(f1_score(y_ma, pred_ma, average='weighted'))
    imp_code = list(le_ma.classes_).index(PRIMARY_ART['imp_label'])
    f1i_ma = float(f1_score(y_ma == imp_code, pred_ma == imp_code))
    ma_df = pd.DataFrame([
        {'mode': 'v1-corregida (fillna=0)', 'n': 612, 'k': ALL_RESULTS['v1-corregida']['baseline']['n_clusters'],
         'sil_UMAP': round(ALL_RESULTS['v1-corregida']['baseline']['sil_umap'], 3),
         'sil_FEAT': round(ALL_RESULTS['v1-corregida']['baseline']['sil_feat'], 3),
         'F1w_strat': _b4_val(ALL_RESULTS['v1-corregida'], 'structural only', 'stratified', 'F1_weighted'),
         'F1i_strat': _b4_val(ALL_RESULTS['v1-corregida'], 'structural only', 'stratified', 'F1_imperial')},
        {'mode': 'missing-aware (NaN + DOC)', 'n': len(F_na), 'k': int(len(set(lab_ma[lab_ma >= 0]))),
         'sil_UMAP': round(su_ma, 3), 'sil_FEAT': round(sf_ma, 3),
         'F1w_strat': round(f1w_ma, 3), 'F1i_strat': round(f1i_ma, 3)},
    ])
    ma_df.to_csv(os.path.join(RESULTS_DIR, 'table_missing_aware_comparison.csv'), index=False)
    print(ma_df.to_string(index=False))


DB: C:\Users\maria\.gemini\antigravity\scratch\Khipu-ML\data\open-khipu-repository-2.1.0\data\khipu.db

===== Run v1-exact =====


[resolve_roles] Detected schema: v1
  twist_ratio     (1): ['ratio_SZ']
  twist_and_dir   (6): ['twist_S', 'twist_Z', 'ratio_SZ', 'knot_dir_S', 'knot_dir_Z', 'ratio_knot_SZ']
  geometry        (7): ['n_cords', 'mean_length', 'std_length', 'max_cord_level', 'ratio_subsidiary', 'n_knots', 'mean_turns']
  size            (2): ['n_cords', 'n_knots']
[v1-exact] n=619, p=27, meta_cov=100.0%, doc_cov=98.9%


[v1-exact] Baseline clusters: {0: 17, 1: 159, 2: 443} | sil_UMAP=0.817 | sil_FEAT=0.062
[v1-exact] Reproduces v1 within UMAP tolerance (got [17, 159, 443], sil_UMAP=0.817, sil_FEAT=0.062)

[P1.2] Crosstab (Inka, Late Horizon vs MUSEUM_NAME):
region                                  False  True 
museum_name                                         
Dallas Museum of Art                        0     18
                                            1      0
Centro Mallqui, Leymebamba, Amazonas       22      0
Ethnologisches Museum, Berlin, Germany     44      0
Musee Quai Branly, Paris, France            4      0
Museo Chileno de Arte Precolombino          1      0



===== Run v1-corregida =====


[resolve_roles] Detected schema: v1
  twist_ratio     (1): ['ratio_SZ']
  twist_and_dir   (6): ['twist_S', 'twist_Z', 'ratio_SZ', 'knot_dir_S', 'knot_dir_Z', 'ratio_knot_SZ']
  geometry        (7): ['n_cords', 'mean_length', 'std_length', 'max_cord_level', 'ratio_subsidiary', 'n_knots', 'mean_turns']
  size            (2): ['n_cords', 'n_knots']
[v1-corregida] n=612, p=27, meta_cov=100.0%, doc_cov=100.0%


[v1-corregida] Baseline clusters: {0: 10, 1: 160, 2: 442} | sil_UMAP=0.793 | sil_FEAT=0.068


... (2 outputs left out)

[robustez] Baseline clusters: {-1: 2, 0: 154, 1: 408, 2: 10, 3: 12, 4: 26} | sil_UMAP=0.780 | sil_FEAT=0.081



================ SENSITIVITY MATRIX ================
     Corrida  Features   n               Clusters  Sil_UMAP  Sil_FEAT  B1_ARI  B3_V_cat  B3_V_mus  B3_doc_acc  A1_lbl  B4_F1w_strat  B4_F1i_strat  B4_F1w_cat  B4_F1w_mus  B5_n_imp  B5_all_F1  B5_geo_F1  B5_doc_F1  B6_dF1_strat
    v1-exact        27 619         [17, 159, 443]     0.817     0.062    0.97      0.79      0.31        0.95     120         0.459         0.833       0.008       0.006        18      0.824      0.824      0.914        -0.009
v1-corregida        27 612         [10, 160, 442]     0.793     0.068    0.90      0.73      0.30        0.95     113         0.465         0.762       0.007       0.007        11      0.778      0.778      0.900        -0.010
    robustez        23 612 [10, 12, 26, 154, 408]     0.780     0.081    0.98      0.50      0.23        0.87     113         0.443         0.762       0.023       0.007        11      0.778      0.778      0.900        -0.025

===== P2.2 Cord-level TWIST by cluste

tw_cat     S     U     Z  blank_or_null
cl                                     
0        0.0  14.7   0.0           85.3
1        1.5  68.1  27.6            2.8
2       90.1   6.2   2.7            1.0

===== P2.4 Missing-aware mode (NaN + completeness indicators) vs fillna(0) =====


                     mode   n  k  sil_UMAP  sil_FEAT  F1w_strat  F1i_strat
  v1-corregida (fillna=0) 612  3     0.793     0.068      0.465      0.762
missing-aware (NaN + DOC) 612  4     0.650     0.198      0.582      0.900


---
## 4. Section B7 — `CREATED_BY = jclindaniel` Audit & Clean-Corpus Reclustering

Orden secuencial de filtros en B7 (OKR Zenodo `2.1.0`):
1. `empty_no_cords` (`n = 7`)
2. `pct_twist_U_ge_0.20` (`n = 145`)
3. `created_by_jclindaniel` (`n = 37`)

> **Nota metodológica:** los registros de `jclindaniel` se tratan como torsión no comparable (sin recodificar `S`/`Z`). La causa del contraste S/Z observado en Incahuasi y Armatambo se mantiene como **hipótesis pendiente; consulta enviada a OKR**.


In [6]:
# ===================== B7 CONFIG =====================
B7_SEED          = 42
B7_N_SIM         = 1000
B7_N_WORKERS     = min(6, os.cpu_count() or 1)
B7_RUN_NULLS     = False     # Set True to recompute all 1,000+1,000 nulls; False loads results/table_b2_1000.csv
B7_ZENODO_VER    = OKR_VERSION
B7_ZENODO_DOI    = f'10.5281/zenodo.{ZENODO_ID}'
B7_EXPECTED_META = ['okr_num', 'investigator_num', 'created_by', 'museum_name', 'provenance', 'region']
B7_EXPECTED_FEAT = ['n_cords', 'ratio_SZ', 'twist_S', 'twist_Z', 'n_knots', 'knot_dir_S', 'knot_dir_Z', 'ratio_knot_SZ']
B7_EXPECTED_DOC  = ['pct_twist_Ucode', 'pct_twist_blank', 'pct_twist_U', 'pct_no_length', 'color_rows_per_cord', 'knot_rows_per_cord', 'pct_knot_dir_U', 'pct_side_U']
# =====================================================

F619_b7, DOC619_b7, META619_b7, _, roles_b7 = load_dataset('v1-exact', con, verbose=False)
assert not [c for c in B7_EXPECTED_META if c not in META619_b7.columns], 'Missing B7 metadata cols'
assert not [c for c in B7_EXPECTED_FEAT if c not in F619_b7.columns], 'Missing B7 feature cols'
assert not [c for c in B7_EXPECTED_DOC  if c not in DOC619_b7.columns], 'Missing B7 DOC cols'
print(f'[B7 Config] OK | OKR Zenodo {B7_ZENODO_VER} | n={len(F619_b7)} | B7_N_SIM={B7_N_SIM} | workers={B7_N_WORKERS} | B7_RUN_NULLS={B7_RUN_NULLS}')


[B7 Config] OK | OKR Zenodo 2.1.0 | n=619 | B7_N_SIM=1000 | workers=6 | B7_RUN_NULLS=False


In [7]:
with step('B7.1 jclindaniel audit by site and CREATED_BY'):
    q_sites = '''
    SELECT
        CASE
            WHEN k.PROVENANCE LIKE '%Incahuasi%' OR k.MUSEUM_NAME LIKE '%Incahu%' THEN 'Incahuasi'
            WHEN k.PROVENANCE LIKE '%Armatambo%' OR k.PROVENANCE LIKE '%Huaca San Pedro%' THEN 'Armatambo / Huaca San Pedro'
        END AS site,
        SUBSTR(k.INVESTIGATOR_NUM, 1, 2) AS cataloger_prefix,
        k.CREATED_BY,
        COUNT(DISTINCT k.KHIPU_ID) AS n_khipus,
        SUM(CASE WHEN UPPER(TRIM(COALESCE(c.TWIST,'')))='S' THEN 1 ELSE 0 END) AS cords_S,
        SUM(CASE WHEN UPPER(TRIM(COALESCE(c.TWIST,'')))='Z' THEN 1 ELSE 0 END) AS cords_Z,
        SUM(CASE WHEN UPPER(TRIM(COALESCE(c.TWIST,''))) NOT IN ('S','Z') THEN 1 ELSE 0 END) AS cords_U
    FROM khipu_main k JOIN cord c ON k.KHIPU_ID = c.KHIPU_ID
    WHERE k.PROVENANCE LIKE '%Incahuasi%' OR k.MUSEUM_NAME LIKE '%Incahu%'
       OR k.PROVENANCE LIKE '%Armatambo%' OR k.PROVENANCE LIKE '%Huaca San Pedro%'
    GROUP BY site, cataloger_prefix, k.CREATED_BY
    ORDER BY site, n_khipus DESC
    '''
    b7_sites_df = pd.read_sql(q_sites, con)
    b7_sites_df.to_csv(os.path.join(RESULTS_DIR, 'table_b7_jclindaniel_sites.csv'), index=False)
    print(b7_sites_df.to_string(index=False))
    print('\nNota: contraste S/Z observado; causa S<->Z = hipotesis pendiente; consulta enviada a OKR.')



===== B7.1 jclindaniel audit by site and CREATED_BY =====
                       site cataloger_prefix  CREATED_BY  n_khipus  cords_S  cords_Z  cords_U
Armatambo / Huaca San Pedro               UR jclindaniel        13      117      620        1
Armatambo / Huaca San Pedro               HP       julia         1       32        0        2
                  Incahuasi               UR       julia        28     4501        0        0
                  Incahuasi               JC jclindaniel        23        1     1214        2
                  Incahuasi               UR      gurton         1       14        0        0

Nota: contraste S/Z observado; causa S<->Z = hipotesis pendiente; consulta enviada a OKR.


In [8]:
with step('B7.2 Export excluded_khipus_okr_2.1.0.csv'):
    is_empty = F619_b7['n_cords'] == 0
    is_unrec = (~is_empty) & (DOC619_b7['pct_twist_U'] >= 0.20)
    is_jc    = (~is_empty) & (~is_unrec) & META619_b7['created_by'].fillna('').str.lower().str.contains('clindaniel')

    reason = pd.Series('', index=F619_b7.index)
    reason.loc[is_empty] = 'empty_no_cords'
    reason.loc[is_unrec] = 'pct_twist_U_ge_0.20'
    reason.loc[is_jc]    = 'created_by_jclindaniel'

    excl_df = pd.DataFrame({
        'KHIPU_ID': F619_b7.index, 'OKR_NUM': META619_b7['okr_num'].values,
        'INVESTIGATOR_NUM': META619_b7['investigator_num'].values, 'CREATED_BY': META619_b7['created_by'].values,
        'MUSEUM_NAME': META619_b7['museum_name'].values, 'PROVENANCE': META619_b7['provenance'].values,
        'REGION': META619_b7['region'].values, 'n_cords': F619_b7['n_cords'].astype(int).values,
        'pct_twist_Ucode': DOC619_b7['pct_twist_Ucode'].round(4).values,
        'pct_twist_blank': DOC619_b7['pct_twist_blank'].round(4).values,
        'pct_twist_U': DOC619_b7['pct_twist_U'].round(4).values,
        'ratio_SZ': F619_b7['ratio_SZ'].round(4).values,
        'exclusion_reason': reason.values,
        'okr_zenodo_version': B7_ZENODO_VER, 'okr_zenodo_doi': B7_ZENODO_DOI,
    })
    excl_only = excl_df[excl_df['exclusion_reason'] != ''].copy()
    excl_path = os.path.join(RESULTS_DIR, f'excluded_khipus_okr_{B7_ZENODO_VER}.csv')
    excl_only.to_csv(excl_path, index=False)
    print(f'Total={len(F619_b7)} | Excluded={len(excl_only)} | Clean remaining={len(F619_b7) - len(excl_only)}')
    print(excl_only['exclusion_reason'].value_counts().to_string())



===== B7.2 Export excluded_khipus_okr_2.1.0.csv =====
Total=619 | Excluded=189 | Clean remaining=430
exclusion_reason
pct_twist_U_ge_0.20       145
created_by_jclindaniel     37
empty_no_cords              7


In [9]:
with step('B7.3 Progression 612 / 467 / 430, DOC Depth-2 Tree & Threshold Sweep'):
    F612_b7 = F619_b7.loc[~is_empty].copy()
    F467_b7 = F619_b7.loc[(~is_empty) & (~is_unrec)].copy()
    F430_b7 = F619_b7.loc[(~is_empty) & (~is_unrec) & (~is_jc)].copy()
    B7_SUBSETS = [
        ('n=612 (v1-corregida)',             F612_b7),
        ('n=467 (recorded <20% U)',          F467_b7),
        ('n=430 (recorded, no jclindaniel)', F430_b7),
    ]
    # P1.2: Imperial count in n=430
    n_imp_430 = int((PRIMARY_ART['nonnull'].reindex(F430_b7.index) == PRIMARY_ART['imp_label']).sum())
    print(f"[P1.2] '{PRIMARY_ART['imp_label']}' en n=430: {n_imp_430} (1 museo -> clase no evaluable en n=430)")

    CAT619 = META619_b7['investigator_num'].astype(str).str.extract(r'^([A-Za-z]{2})')[0].str.upper().fillna('NA')
    doc_tree_cols = ['pct_twist_Ucode', 'pct_twist_blank', 'pct_no_length', 'color_rows_per_cord', 'knot_rows_per_cord', 'pct_knot_dir_U', 'pct_side_U']

    prog_rows, tree_rows = [], []
    for name, Fsub in [('n=619 (v1-exact)', F619_b7)] + B7_SUBSETS:
        Z = StandardScaler().fit_transform(Fsub.values)
        _, lab, su, sf = run_cluster(Z, seed=B7_SEED)
        ok = lab >= 0; k = int(len(set(lab[ok])))
        noise_pct = round(100.0 * float(np.mean(lab < 0)), 1)
        v_cat = cramers_v(pd.crosstab(pd.Series(lab, index=Fsub.index)[ok], CAT619.loc[Fsub.index][ok])) if k >= 2 else np.nan
        # P1.4: Depth-2 Decision Tree using ONLY DOC to predict clusters
        Xd = DOC619_b7.loc[Fsub.index, doc_tree_cols].values[ok]; yd = lab[ok]
        maj = float(pd.Series(yd).value_counts(normalize=True).max())
        dt = DecisionTreeClassifier(max_depth=2, random_state=B7_SEED)
        acc_dt = float(cross_val_score(dt, Xd, yd, cv=5).mean()) if pd.Series(yd).value_counts().min() >= 5 else np.nan
        dt.fit(Xd, yd)
        top_doc = doc_tree_cols[int(np.argmax(dt.feature_importances_))]
        tree_rows.append(dict(subset=name, n=len(Fsub), k=k, sil_UMAP=round(su, 3), sil_FEAT=round(sf, 3),
                              doc_tree_d2_acc=round(acc_dt, 3), majority=round(maj, 3), top_doc_feature=top_doc))
        if name != 'n=619 (v1-exact)':
            F_nt = Fsub.drop(columns=roles_b7['twist_and_dir'])
            _, lab_nt, su_nt, sf_nt = run_cluster(StandardScaler().fit_transform(F_nt.values), seed=B7_SEED)
            prog_rows.append({
                'subset': name, 'n': len(Fsub), 'k': k, 'sil_UMAP': round(su, 3), 'sil_FEAT': round(sf, 3),
                'noise_pct': noise_pct, 'cramers_v_cat': round(v_cat, 3),
                'ratio_SZ_mean': round(float(Fsub['ratio_SZ'].mean()), 4),
                'ratio_SZ_var': round(float(Fsub['ratio_SZ'].var()), 4),
                'no_td21_k': int(len(set(lab_nt[lab_nt >= 0]))),
                'no_td21_sil_UMAP': round(su_nt, 3), 'no_td21_sil_FEAT': round(sf_nt, 3),
                'no_td21_noise_pct': round(100.0 * float(np.mean(lab_nt < 0)), 1),
            })

    pd.DataFrame(prog_rows).to_csv(os.path.join(RESULTS_DIR, 'table_b7_progression.csv'), index=False)
    pd.DataFrame(tree_rows).to_csv(os.path.join(RESULTS_DIR, 'table_b7_doc_tree.csv'), index=False)
    print('\nProgression 612 / 467 / 430:')
    print(pd.DataFrame(prog_rows).to_string(index=False))
    print('\n[P1.4] Depth-2 Decision Tree on DOC only:')
    print(pd.DataFrame(tree_rows).to_string(index=False))

    # P2.3: Exclusion threshold sweep (0.05 / 0.10 / 0.20 / 0.50) without jclindaniel
    is_jc_all = META619_b7['created_by'].fillna('').str.lower().str.contains('clindaniel')
    th_rows = []
    for th in [0.05, 0.10, 0.20, 0.50]:
        keep = (~is_empty) & (DOC619_b7['pct_twist_U'] < th) & (~is_jc_all)
        Zt = StandardScaler().fit_transform(F619_b7.loc[keep].values)
        _, lt, su_t, sf_t = run_cluster(Zt, seed=B7_SEED)
        th_rows.append(dict(threshold=th, n=int(keep.sum()), k=int(len(set(lt[lt >= 0]))),
                            sil_UMAP=round(su_t, 3), sil_FEAT=round(sf_t, 3), noise_pct=round(100.0 * float(np.mean(lt < 0)), 1)))
    th_df = pd.DataFrame(th_rows)
    th_df.to_csv(os.path.join(RESULTS_DIR, 'table_b7_threshold_sweep.csv'), index=False)
    print('\n[P2.3] Exclusion threshold sweep (no jclindaniel):')
    print(th_df.to_string(index=False))



===== B7.3 Progression 612 / 467 / 430, DOC Depth-2 Tree & Threshold Sweep =====
[P1.2] 'Inka, Late Horizon' en n=430: 3 (1 museo -> clase no evaluable en n=430)



Progression 612 / 467 / 430:
                          subset   n  k  sil_UMAP  sil_FEAT  noise_pct  cramers_v_cat  ratio_SZ_mean  ratio_SZ_var  no_td21_k  no_td21_sil_UMAP  no_td21_sil_FEAT  no_td21_noise_pct
            n=612 (v1-corregida) 612  3     0.793     0.068        0.0          0.733         0.6946        0.2087          2             0.807             0.519                0.0
         n=467 (recorded <20% U) 467  2     0.591     0.192        0.0          0.756         0.8942        0.0909         11             0.446             0.025               29.3
n=430 (recorded, no jclindaniel) 430  3     0.210     0.107        3.3          0.174         0.9700        0.0258         11             0.381            -0.035               17.7

[P1.4] Depth-2 Decision Tree on DOC only:
                          subset   n  k  sil_UMAP  sil_FEAT  doc_tree_d2_acc  majority     top_doc_feature
                n=619 (v1-exact) 619  3     0.817     0.062            0.914     0.716     pct_t


[P2.3] Exclusion threshold sweep (no jclindaniel):
 threshold   n  k  sil_UMAP  sil_FEAT  noise_pct
      0.05 395  8     0.415    -0.022       19.7
      0.10 420 10     0.451     0.002       27.4
      0.20 430  3     0.210     0.107        3.3
      0.50 434  3     0.082     0.125        8.3


In [10]:
with step('B7.4 Hyperparameter sweep (18 configs per subset, UMAP + FEAT sil)'):
    sweep_rows = []
    for name, Fsub in B7_SUBSETS:
        Z = StandardScaler().fit_transform(Fsub.values)
        for nn in [10, 15, 30]:
            for md in [0.0, 0.1]:
                emb = umap.UMAP(n_components=2, n_neighbors=nn, min_dist=md, metric='euclidean', random_state=B7_SEED).fit_transform(Z)
                for mcs in [10, 15, 25]:
                    lab = hdbscan.HDBSCAN(min_cluster_size=mcs, min_samples=5, metric='euclidean').fit_predict(emb)
                    ok = lab >= 0; k = int(len(set(lab[ok])))
                    su = float(silhouette_score(emb[ok], lab[ok])) if ok.sum() > 1 and k > 1 else np.nan
                    sf = sil_feat(Z, lab)
                    sweep_rows.append(dict(
                        subset=name, n=len(Fsub), n_neighbors=nn, min_dist=md, min_cluster_size=mcs,
                        k=k, sil_UMAP=round(su, 3) if not np.isnan(su) else np.nan,
                        sil_FEAT=round(sf, 3) if not np.isnan(sf) else np.nan,
                        noise_pct=round(100.0 * float(np.mean(lab < 0)), 1),
                    ))
    B7_SWEEP_DF = pd.DataFrame(sweep_rows)
    B7_SWEEP_DF.to_csv(os.path.join(RESULTS_DIR, 'table_hyperparam_sweep.csv'), index=False)
    print(B7_SWEEP_DF.groupby('subset').agg(
        k_min=('k', 'min'), k_med=('k', 'median'), k_max=('k', 'max'),
        silU_med=('sil_UMAP', 'median'), silF_med=('sil_FEAT', 'median'), silF_max=('sil_FEAT', 'max'),
        noise_med=('noise_pct', 'median'),
    ).round(3).to_string())



===== B7.4 Hyperparameter sweep (18 configs per subset, UMAP + FEAT sil) =====


                                  k_min  k_med  k_max  silU_med  silF_med  silF_max  noise_med
subset                                                                                        
n=430 (recorded, no jclindaniel)      2    3.0     13     0.380     0.106     0.135       11.4
n=467 (recorded <20% U)               2    2.0      2     0.619     0.192     0.205        0.0
n=612 (v1-corregida)                  2    3.0      7     0.756     0.045     0.072        1.6


In [11]:
with step('B7.5 B2 null models (N_SIM=1000+1000, UMAP & feature space)'):
    b2_path = os.path.join(RESULTS_DIR, 'table_b2_1000.csv')
    if (not B7_RUN_NULLS) and os.path.exists(b2_path):
        B7_NULL_DF = pd.read_csv(b2_path)
        print(f'Loaded cached B2 results from {b2_path} (set B7_RUN_NULLS=True to recompute):')
        print(B7_NULL_DF.to_string(index=False))
    else:
        b2_rows = []
        for s_idx, (name, Fsub) in enumerate(B7_SUBSETS):
            Z = StandardScaler().fit_transform(Fsub.values)
            _, lab_obs, su_obs, sf_obs = run_cluster(Z, seed=B7_SEED)
            k_obs = int(len(set(lab_obs[lab_obs >= 0])))
            row = {'subset': name, 'n': len(Fsub), 'k_obs': k_obs,
                   'sil_umap_obs': round(su_obs, 3), 'sil_feat_obs': round(sf_obs, 3)}
            for k_idx, kind in enumerate(['perm', 'gauss']):
                base_s = 100000 * (s_idx + 1) + 10000 * (k_idx + 1)
                arr = run_null_suite(Z, kind, B7_N_SIM, base_s, UMAP_KW, HDB_KW, n_workers=B7_N_WORKERS)
                cnt_u = int(np.nansum(arr[:, 1] >= su_obs))
                cnt_f = int(np.nansum(arr[:, 2] >= sf_obs))
                row[f'{kind}_umap_mean'] = round(float(np.nanmean(arr[:, 1])), 3)
                row[f'{kind}_umap_p95']  = round(float(np.nanpercentile(arr[:, 1], 95)), 3)
                row[f'{kind}_umap_p']    = format_p(cnt_u, B7_N_SIM)
                row[f'{kind}_feat_mean'] = round(float(np.nanmean(arr[:, 2])), 3)
                row[f'{kind}_feat_p95']  = round(float(np.nanpercentile(arr[:, 2], 95)), 3)
                row[f'{kind}_feat_p']    = format_p(cnt_f, B7_N_SIM)
            b2_rows.append(row)
        B7_NULL_DF = pd.DataFrame(b2_rows)
        B7_NULL_DF.to_csv(b2_path, index=False)
        print(B7_NULL_DF.to_string(index=False))



===== B7.5 B2 null models (N_SIM=1000+1000, UMAP & feature space) =====
Loaded cached B2 results from C:\Users\maria\.gemini\antigravity\scratch\Khipu-ML\results\table_b2_1000.csv (set B7_RUN_NULLS=True to recompute):
                          subset   n  k_obs  sil_umap_obs  sil_feat_obs  perm_umap_mean  perm_umap_p95  perm_umap_p  perm_feat_mean  perm_feat_p95  perm_feat_p  gauss_umap_mean  gauss_umap_p95 gauss_umap_p  gauss_feat_mean  gauss_feat_p95  gauss_feat_p
n=430 (recorded, no jclindaniel) 430      3         0.210         0.107           0.493          0.763       0.9131           0.262          0.361       0.9261            0.327           0.476       0.8142            0.044           0.089         0.043
         n=467 (recorded <20% U) 467      2         0.591         0.192           0.412          0.622       0.1029           0.077          0.138       0.0090            0.326           0.471   p <= 0.001            0.037           0.074         0.002
            n=612 (v1-

---
## 5. Santa Valley Replication (Clean Query by `OKR_NUM`) & Exports


In [12]:
with step('Santa Valley replication (OKR_NUM BETWEEN KH0323 AND KH0328)'):
    sv_main = pd.read_sql('''
        SELECT KHIPU_ID, OKR_NUM, INVESTIGATOR_NUM, MUSEUM_NAME
        FROM khipu_main
        WHERE OKR_NUM BETWEEN 'KH0323' AND 'KH0328'
        ORDER BY OKR_NUM
    ''', con)
    assert len(sv_main) == 6, f'Expected 6 Santa Valley khipus, got {len(sv_main)}'
    sv_cords = pd.read_sql('''
        SELECT k.OKR_NUM, c.ATTACHMENT_TYPE
        FROM cord c JOIN khipu_main k ON c.KHIPU_ID = k.KHIPU_ID
        WHERE k.OKR_NUM BETWEEN 'KH0323' AND 'KH0328'
    ''', con)
    sv_rv = pd.crosstab(sv_cords['OKR_NUM'], sv_cords['ATTACHMENT_TYPE']).reindex(columns=['R', 'V'], fill_value=0)
    sv_rv.to_csv(os.path.join(RESULTS_DIR, 'table_santa_valley_rv.csv'))
    tot_r, tot_v = int(sv_rv['R'].sum()), int(sv_rv['V'].sum())
    print(sv_rv.to_string())
    print(f'Aggregate R/V: R={tot_r} ({100*tot_r/(tot_r+tot_v):.1f}%), V={tot_v} ({100*tot_v/(tot_r+tot_v):.1f}%)')

with step('Export full JSON'):
    with open(os.path.join(OUTPUT_DIR, 'okr_v2_results.json'), 'w', encoding='utf-8') as f:
        json.dump(ALL_RESULTS, f, indent=2, default=str)

print('\n================ STEP STATUS ================')
for k, s in STATUS.items():
    print(f'{k:55s} {s}')
assert all(v == 'OK' for v in STATUS.values()), f'Some steps failed: {STATUS}'



===== Santa Valley replication (OKR_NUM BETWEEN KH0323 AND KH0328) =====
ATTACHMENT_TYPE    R    V
OKR_NUM                  
KH0323           310    0
KH0324             0   54
KH0325             0  207
KH0326            40   69
KH0327             0   93
KH0328            56    0
Aggregate R/V: R=406 (49.0%), V=423 (51.0%)

===== Export full JSON =====

================ STEP STATUS ================
Run v1-exact                                            OK
Run v1-corregida                                        OK
Run robustez                                            OK
P2.2 Cord-level TWIST by cluster (correcting v1 85.3%)  OK
P2.4 Missing-aware mode (NaN + completeness indicators) vs fillna(0) OK
B7.1 jclindaniel audit by site and CREATED_BY           OK
B7.2 Export excluded_khipus_okr_2.1.0.csv               OK
B7.3 Progression 612 / 467 / 430, DOC Depth-2 Tree & Threshold Sweep OK
B7.4 Hyperparameter sweep (18 configs per subset, UMAP + FEAT sil) OK
B7.5 B2 null models (N_SIM=10